In [1]:
import pandas as pd
import pulp

In [2]:
s_df = pd.read_csv('students.csv')
s_df.head()

,student_id,gender,leader_flag,support_flag,score
0,1,0,0,0,335
1,2,1,0,0,379
2,3,0,0,0,350
3,4,0,0,0,301
4,5,1,0,0,317


In [8]:
s_pair_df = pd.read_csv('student_pairs.csv')
s_pair_df


,student_id1,student_id2
0,118,189
1,72,50
2,314,233


In [9]:
prob = pulp.LpProblem('ClassAssignmentProblem', pulp.LpMinimize)

In [11]:
S = s_df['student_id'].to_list()
C = ["A", 'B', 'C', 'D', 'E', 'F', 'G', 'H']

In [14]:
SC = [(s, c) for s in S for c in C]

x = pulp.LpVariable.dicts('x', SC, cat='Binary')


In [15]:
for s in S:
    prob += pulp.lpSum([x[s, c] for c in C]) == 1

In [16]:
for c in C:
    prob += pulp.lpSum([x[s, c] for s in S]) >= 39
    prob += pulp.lpSum([x[s, c] for s in S]) <= 40

In [ ]:
S_male = [row.student_id for row in s_df.itertuples() if row.gender == 1]
S_female = [row.student_id for row in s_df.itertuples() if row.gender == 0]

In [20]:
for c in C:
    prob += pulp.lpSum([x[s, c] for s in S_male]) <= 20
    prob += pulp.lpSum([x[s, c] for s in S_female]) <= 20

In [22]:
score = {row.student_id:row.score for row in s_df.itertuples()}

score_mean = s_df['score'].mean()

for c in C:
    prob += (score_mean - 10) * pulp.lpSum([x[s, c] for s in S]) <= pulp.lpSum([x[s, c] * score[s] for s in S])
    prob += pulp.lpSum([x[s, c] * score[s] for s in S]) <= (score_mean + 10) * pulp.lpSum([x[s, c] for s in S])

In [23]:
# リーダー気質の生徒の集合
S_leader = [row.student_id for row in s_df.itertuples() if row.leader_flag == 1]

# (5)各クラスにリーダー気質の生徒を2人以上割り当てる。
for c in C:
    prob += pulp.lpSum([x[s,c] for s in S_leader]) >= 2

In [24]:
# 特別な支援が必要な生徒の集合
S_support = [row.student_id for row in s_df.itertuples() if row.support_flag == 1]

# (6) 特別な支援が必要な生徒は各クラスに1人以下とする。
for c in C:
    prob += pulp.lpSum([x[s,c] for s in S_support]) <= 1

In [28]:
SS = [(row.student_id1, row.student_id2) for row in s_pair_df.itertuples()]
for s1, s2 in SS:
    for c in C:
        prob += x[s1, c] + x[s2, c] <= 1

In [29]:
status = prob.solve()
print(status)
print(pulp.LpStatus[status])

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/miyakoh/dev/tech_books/PyOptBook-main/.venv/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/fk/j0dk8tns78g4y3w9yv61yf780000gn/T/43a1748a4b9f455ebdebea7be8552897-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/fk/j0dk8tns78g4y3w9yv61yf780000gn/T/43a1748a4b9f455ebdebea7be8552897-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 411 COLUMNS
At line 20981 RHS
At line 21388 BOUNDS
At line 23934 ENDATA
Problem MODEL has 406 rows, 2545 columns and 15480 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0 - 0.05 seconds
Cgl0005I 318 SOS with 2544 members
Cgl0004I processed model has 398 rows, 2544 columns (2544 integer (2544 of which binary)) and 12936 elements
Cbc0038I Initial state - 38 integers unsatisfied sum - 7.12529
Cbc0038I Pass   1:

In [30]:
C2Ss = {}
for c in C:
    C2Ss[c] = [s for s in S if x[s, c].value()==1]

for c, Ss in C2Ss.items():
    print('Class:', c)
    print('Num:', len(Ss))
    print('Student:', Ss)
    print()

Class: A
Num: 39
Student: [2, 9, 19, 39, 42, 63, 65, 71, 79, 83, 85, 88, 99, 109, 111, 123, 126, 136, 138, 145, 148, 165, 168, 173, 177, 179, 180, 193, 199, 206, 224, 233, 240, 246, 264, 267, 291, 292, 298]

Class: B
Num: 40
Student: [11, 15, 43, 48, 50, 70, 82, 89, 91, 102, 104, 113, 114, 120, 121, 124, 127, 146, 149, 159, 167, 170, 172, 176, 190, 203, 213, 220, 222, 231, 238, 245, 263, 270, 275, 276, 283, 287, 290, 317]

Class: C
Num: 40
Student: [3, 14, 23, 27, 31, 33, 41, 49, 53, 54, 58, 73, 93, 97, 98, 107, 122, 152, 156, 160, 171, 187, 201, 210, 211, 217, 219, 227, 236, 242, 254, 258, 260, 268, 273, 274, 277, 278, 301, 318]

Class: D
Num: 40
Student: [10, 13, 16, 36, 37, 51, 56, 59, 61, 67, 68, 75, 84, 92, 108, 128, 134, 139, 140, 158, 161, 175, 183, 188, 192, 198, 200, 205, 221, 225, 235, 241, 252, 255, 256, 257, 261, 266, 293, 316]

Class: E
Num: 39
Student: [4, 5, 21, 22, 24, 38, 69, 72, 78, 87, 96, 105, 106, 115, 129, 132, 141, 143, 150, 154, 164, 166, 184, 189, 195, 196, 197

In [31]:
for s in S:
    assigned_class = [c for c in C if x[s, c].value()==1]

    if len(assigned_class) != 1:
        print('error:', s, assigned_class)

In [37]:
result_df = s_df.copy()

S2C = {s:c for s in S for c in C if x[s, c].value()==1}

result_df['assigned_class'] = result_df['student_id'].map(S2C)
result_df.head()

,student_id,gender,leader_flag,support_flag,score,assigned_class
0,1,0,0,0,335,H
1,2,1,0,0,379,A
2,3,0,0,0,350,C
3,4,0,0,0,301,E
4,5,1,0,0,317,E


In [45]:
result_df.groupby('assigned_class')['support_flag'].sum()

assigned_class
A    0
B    1
C    1
D    0
E    1
F    0
G    1
H    0
Name: support_flag, dtype: int64

In [46]:
for i, (s1, s2) in enumerate(SS):
    print('case:', i)
    c1 = S2C[s1]
    c2 = S2C[s2]
    print(f's1:{s1}-{c1}')
    print(f's1:{s2}-{c2}')
    print('')

case: 0
s1:118-G
s1:189-E

case: 1
s1:72-E
s1:50-B

case: 2
s1:314-H
s1:233-A

